# Exercise 5 - Logistic Regression using Keras
**Hardware Accelerator / computer server to use:**

$\begin{array}{lcl}
   \text{Colab/Kaggle} &:& \text{CPU} \\
  \text{CoCalc} &:& \text{Home Server} \\
 \end{array}$

**Suggested duration: ~ 25 minutes**
 
**Goals**:
-  implement the logistic regression algorithm from the slides in Keras
- continue to get you familiar with the structure and the building blocks of Keras code

Once again, this exercise is very easy - you can just copy the code from the slides.

### Dataset Generation
First run the code to generate the dataset. (You can experiment by adjusting the number of datapoints or the random seed, if you want.)

In [ ]:
%matplotlib inline
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Generate the dataset

m          = 99 # The number of  points in the dataset
n_features = 2  # The 2 dimensions of each training data point, coordinate (x,y)
n_labels   = 3  # The categories shown as red, green, and blue

rng = np.random.RandomState(seed=47)  # Seed the random number generator

# draw m points in the plane at random following an uniform distribution
x = np.array((rng.standard_normal(m), rng.standard_normal(m)))
x = np.transpose(x)
y = np.empty((m))

# split all the points in color clusters of roughly the same size
for i in range(n_labels):
    # shift the cluster center for "i-th" color by a random vector (-10..10,-10..10)
    center = rng.randint(-10,+10, (1,2))
    x[i*m//n_labels:(i+1)*m//n_labels] += center
    y[i*m//n_labels:(i+1)*m//n_labels] = i

### Data Preparation

#### One-hot Encoding
The category to which each training sample belongs is indicated using an integer label (0, 1, or 2), but integers are not a good idea because they can bias the results by suggesting a relationship between the categories that does not actually exist (0 < 1 < 2). So it is always a good idea to re-encode the labels using one-hot encoding, i.e. represent n categories by a vector of n elements, where each element is 0 **excepts** for one element which is 1, which corresponds to that particular category.

$
\text{red} = 
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}, \qquad
\text{green} = 
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}, \qquad
\text{blue} = 
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$

In [ ]:
print(f"Before one hot encoding")
print(f"""
y[{0:2}] = {y[0]},
y[{m//n_labels}] = {y[m//n_labels]},
y[{m-1}] = {y[m-1]}
""")

In [ ]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y)

In [ ]:
print(f"After one hot encoding")
print(f"""
y[{0:2}] = {y[0]},
y[{m//n_labels}] = {y[m//n_labels]},
y[{m-1}] = {y[m-1]}
""")

#### Shuffling the data

Due to the order of the samples, all the red points come first, followed by all the green points, and finally all the blue points. This specific order introduces a **bias**. During the training, the machine learning algorithm could push aggressively the trainable parameters towards the red points during the "red points area", only to "unlearn" what it just learned when "green points area" comes, and likewise with the "blue points area". The model could oscillate back-and-forth and takes longer to learn how to categorize the points, or even fail to learn the features that helps to classify the point.

**Another example (time series):** 

Suppose you are training a model to predict if a factory machine is about to fail based on a temperature sensor. You collected data over 24 hours, and (as expected) the factory gets hotter in the afternoon and cooler at night. If you feed the data in chronological order, the model might notice that "higher temperatures" are followed by "even higher temperatures" simply because of the time of day, not because of the machine's health. By shuffling, you break the time-link. Now, a "hot" data point from 2 PM is paired in a batch with a "cold" data point from 3 AM. The Result: The model can no longer rely on the time-of-day trend. It is forced to look at other features—like vibration or pressure—to determine if the high temperature actually indicates a hardware failure.

**Good to know!** 

shuffling is so important that the Keras [model.fit](https://keras.io/api/models/model_training_apis/) will shuffle the data at each epoch per default (parameter `shuffle = True`). If you are using other frameworks, you might need to shuffle the data however.

In [ ]:
indices = np.arange(m)
random.shuffle(indices)
train_x = np.empty((m,n_features)).astype(np.float32)
train_y = np.empty((m,n_labels)).astype(np.float32)

#### Plotting the Training Dataset

In [ ]:
c = []  # List to hold the color of each data point in the plot

# not very pythonic, but easier to understand if not familiar with Python
for i in range(m):
    train_x[i] = x[indices[i]]
    train_y[i] = y[indices[i]]
    if train_y[i,0] == 1:
        c.append('r')
    elif train_y[i,1] == 1:
        c.append('g')
    else:
        c.append('b')

plt.figure(1, figsize=(15, 10))
plt.scatter(train_x[:,0], train_x[:,1], color=c)
plt.show()

### Neural Network Model

Implement the neural network and the gradient descent algorithm for regression classification in TensorFlow. You can use your answer to the previous exercise as a starting point. Print out the cost periodically during gradient descent. Use the predicted values to plot a scatter chart, coloring the points on the chart if the confidence (the softmax probability) exceeds a certain threshold, as shown on the slides. (You can experiment with the threshold value, if you want.)

In [ ]:
#

Run the code in the cell below to plot the decision boundary using a new set of test data distinct from the training data, taking the label that has the highest probability as the predicted output. (Once again, you can experiment with the threshold value, if you want. You can also go back and pick a different random seed.)

In [ ]:
# Test data
test_m = 10000
test_x = (np.random.random((test_m,n_features)).astype(np.float32) - 0.5) * 20.0

prediction = model.predict(test_x, batch_size=test_m)

c = [None for i in range(test_m)]

threshold = 0.95

# Color each test point according to the label with the highest probability
for i in range(test_m):
    x = np.argmax(prediction[i,:])
    c[i] = ('r', 'g', 'b')[x]
    if prediction[i,x] < threshold:
        c[i] = 'y'

plt.scatter(test_x[:,0], test_x[:,1], color=c)
plt.show()

#### Solution 

If you want some help with the answer, you can look at our answer.  Copy and paste necessary sections of the code in a new cell to run the exercise.  Do not click on the cell below unless you want to see the answer we provide!

<details>
    <summary> See our answer </summary>
    
    import tensorflow
    from tensorflow.keras.models     import Sequential
    from tensorflow.keras.layers     import Dense,Input
    from tensorflow.keras.optimizers import SGD

    tensorflow.keras.backend.clear_session()

    model = Sequential()
    model.add(Input(shape=(2,)))
    model.add(Dense(units=3, activation='softmax'))
    model.summary()

    model.compile(loss='categorical_crossentropy', optimizer=SGD(learning_rate=0.05), metrics=['accuracy'])

    n_steps = 5000

    for step in range(n_steps//10,n_steps+1,n_steps//10):
        model.fit(train_x, train_y, epochs=n_steps//10, batch_size=m, verbose=0)
        loss_and_acc = model.evaluate(train_x, train_y, batch_size=m, verbose=0)
        print(f'Loss at step {step:4} = {loss_and_acc[0]:5.3f}, acc = {loss_and_acc[1]:5.3f}')

    prediction = model.predict(train_x, batch_size=m)

    threshold = 0.95

    c = [None]*m

    for i in range(m):
        ix = np.argmax(prediction[i,:])   # The color with the highest probability
        c[i] = ('r', 'g', 'b')[ix]
        if prediction[i,ix] < threshold:
            c[i] = 'y'

    plt.figure(1, figsize=(15, 10))
    plt.scatter(train_x[:,0], train_x[:,1], color=c)
    plt.show()
</details>